In [ ]:
import matplotlib
matplotlib.use("Agg")
import getpass, os, random, gc, random, copy, torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
df_path = os.path.join(script_dir, "Datasets", "Email Spam.csv")
df = pd.read_csv(df_path)
df = df.dropna()
df["target"] = df["target"].map({"Spam":1,"Not_Spam":0})

x = df.drop(columns="target").values
y = df["target"]
x_scaler = StandardScaler()
x = x_scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.79, random_state=41)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

user_account = getpass.getuser()
predictions_bar_path = os.path.join(script_dir, f"Email Spam Predictor Bar Chart.png")

class BinaryClassifer(nn.Module):
    def __init__(self, input_dim=x_train.shape[1]):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=13),
            nn.LeakyReLU(),
            nn.BatchNorm1d(13),

            nn.Linear(in_features=13, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=29, out_features=41),
            nn.LeakyReLU(),
            nn.BatchNorm1d(41),

            nn.Linear(in_features=41, out_features=66),
            nn.LeakyReLU(),
            nn.BatchNorm1d(66),
            nn.Dropout(p=0.6),

            nn.Linear(in_features=66, out_features=85),
            nn.LeakyReLU(),
            nn.BatchNorm1d(85),
            nn.Dropout(p=0.9),

            nn.Linear(in_features=85, out_features=109),
            nn.LeakyReLU(0.45),
            nn.BatchNorm1d(109),
            nn.Dropout(p=0.5),

            nn.Linear(in_features=109, out_features=32),
            nn.LeakyReLU(),
            nn.BatchNorm1d(32),

            nn.Linear(in_features=32, out_features=1)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)            

losses = []
accuracies = []
def grid_search(epoch_count_list, rates_list, seed_val):
    best_state = None
    loss_fn = nn.BCEWithLogitsLoss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15} | {'Val Accuracy':<15}")
    print(f"="*80)

    for lr in rates_list:
        for epoch_count in epoch_count_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = BinaryClassifer().to(device)
            optimizer = torch.opim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch_count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)

                optimizer.zero_grad()   
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_logits = model(x_test)
                test_loss = loss_fn(test_logits, y_test)
                accuracy = BinaryAccuracy().to(device)
                precision = BinaryPrecision().to(device)
                recall = BinaryRecall().to(device)
                test_acc = accuracy(torch.sigmoid(test_logits), y_test).item()
                losses.append(test_loss)
                accuracies.append(test_acc)

            print(f"{lr:<15} | {epoch_count:<15} | {test_loss:<15} | {test_acc:<15}")
            best_state = copy.deepcopy(model.state_dict())
            del model, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies):.2f}")
    print(f"● Worst Loss = {max(losses):.2f} | Worst Accuracy = {min(accuracies):.2f}")
    print(f"● Precision Score = {precision(torch.sigmoid(test_logits), y_preds).item()}")
    print(f"● Recall Score = {recall(torch.sigmoid(test_logits), y_preds).item()}\n")
    return best_state

epoch_counts_list = [900,810,700,1020]
learning_rates = [0.1, 0.01, 0.001, 0.0001]
seed_val = float(input("-- Enter a Value for a Seed : ").strip())
best_state = grid_search(epoch_count_list=epoch_counts_list, rates_list=learning_rates, seed_val=seed_val)
torch.save(best_state, f"Email Spam Predictor Parameters.pth")

fresh_model = BinaryClassifer().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

conversion = {
    1:"Spam",
    0:"Not Spam"
}
while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip()
    if user_name.lower() == "exit" or not user_name:
        print(f"Exiting Interactive Loop ...")
        break

    word_count = float(input('-- Enter the Word Count of the Email : ').strip())
    link_count = float(input("-- Enter the Links Count of the Email : ").strip())
    uppercase_ratio = float(input("-- Enter the Uppercase Ratio : ").strip())
    sender_reputation = float(input("-- Enter the Sender Reputation : ").strip())
    attachment_count = float(input("-- Enter the Attachment Count of the Email : ").strip())

    raw_array = np.array([word_count, link_count, uppercase_ratio, sender_reputation, attachment_count])
    scaled_tensor = torch.tensor(x_scaler.transform(raw_array), dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = fresh_model(scaled_tensor)
        probabilty = torch.sigmoid(logits).item()

    print(f"Predicted Email Spam Status = {conversion.get(probabilty, 0)}")
    session_inputs.append(f"{user_name.title()} ( {word_count} Words )")
    session_predictions.append(probabilty * 100)

try:
    onedrive = input("-- Do you use One Drive or Not : ").strip().title()
    if onedrive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Email Spam Predictor Details.txt"
    elif onedrive == "No":
        desktop_path = f"C:/Users/{user_account}/Desktop/Email Spam Predictor Details.txt"
except BaseException as e:
    print(f"Invalid Input. Error has occured = {e}")

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
        file.write("== Email Spam Predictor Details ==") 
        file.write(f"\n● Parameter Weight = {fresh_model.network[0].weight[0][0].item():.2f}")
        file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.2f}")
        file.write(f"● Best Loss = {min(losses):.3f} | Best Accuracy = {max(accuracies):.3f}")
        file.write(f"● Worst Loss = {max(losses):.3f} | Worst Accuracy = {min(accuracies):.3f}")
    print(f"Successfully Saved your Model Details")
except BaseException as e:
    print(f"Failed to Save Network Details")               
